In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

# 1. Data loading and preparation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_dataset = datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

train_dataset, test_dataset = random_split(full_dataset, [50000, 10000])

# 2. Model definition
class FashionClassifier(nn.Module):
    def __init__(self):
        super(FashionClassifier, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# 3. Evaluate
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return correct / total

# 4. Loss function
criterion = nn.CrossEntropyLoss()

# 5. Experiment setup
batch_sizes = [16, 64, 256]
epochs = 5
results = {}

# 6. Train with different batch sizes
for batch_size in batch_sizes:
    print(f"\nTesting batch size: {batch_size}")
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    model = FashionClassifier()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    loss_history = []
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        avg_loss = running_loss / len(train_loader)
        loss_history.append(avg_loss)
        
        print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    
    acc = evaluate(model, test_loader)
    print(f"Accuracy (batch {batch_size}): {acc:.4f}")
    
    results[batch_size] = {
        "loss": loss_history,
        "accuracy": acc
    }

# 7. Visualization
for batch_size in batch_sizes:
    plt.plot(results[batch_size]["loss"], label=f"batch {batch_size}")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Batch Size Comparison")
plt.legend()
plt.show()


Testing batch size: 16
